In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder


In [2]:
SEED = 143

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

In [3]:
path = "data/ObesityDataSet_raw_and_data_sinthetic.csv"
df = pd.read_csv(path)

df = df.drop_duplicates()  # см. EDA

df.shape, df.head()

((2087, 17),
     Age  Gender  Height  Weight        CALC FAVC  FCVC  NCP  SCC SMOKE  CH2O  \
 0  21.0  Female    1.62    64.0          no   no   2.0  3.0   no    no   2.0   
 1  21.0  Female    1.52    56.0   Sometimes   no   3.0  3.0  yes   yes   3.0   
 2  23.0    Male    1.80    77.0  Frequently   no   2.0  3.0   no    no   2.0   
 3  27.0    Male    1.80    87.0  Frequently   no   3.0  3.0   no    no   2.0   
 4  22.0    Male    1.78    89.8   Sometimes   no   2.0  1.0   no    no   2.0   
 
   family_history_with_overweight  FAF  TUE       CAEC                 MTRANS  \
 0                            yes  0.0  1.0  Sometimes  Public_Transportation   
 1                            yes  3.0  0.0  Sometimes  Public_Transportation   
 2                            yes  2.0  1.0  Sometimes  Public_Transportation   
 3                             no  2.0  0.0  Sometimes                Walking   
 4                             no  0.0  0.0  Sometimes  Public_Transportation   
 
           

In [4]:
target = "NObeyesdad"

X = df.drop(columns=[target])
y = df[target]

features = X.columns.to_list()
numerical_features = X.select_dtypes("float64").columns.to_list()
str_features = X.select_dtypes("str").columns.to_list()


In [5]:
transformer = ColumnTransformer(
    transformers=[
        (
            "nums",
            StandardScaler(),
            make_column_selector(dtype_include=np.float64),
        ),
        (
            "strs",
            OneHotEncoder(handle_unknown="ignore"),
            make_column_selector(dtype_include=object),
        ),
    ]
)  # KNN'у нужен скейлер


In [6]:
pipeline = make_pipeline(
    transformer,
    KNeighborsClassifier(),
)


### 1. Рост и вес содержат достаточно информации для классификации

**Гипотеза:** KNN только на `Height` и `Weight` покажет качество не ниже KNN на всех исходных признаках.


In [7]:
df1 = df.copy()

In [8]:
X1 = df1[["Height", "Weight"]]

only2features = cross_val_score(pipeline, X1, y, cv=CV, scoring="f1_macro")
only2features_accuracy = cross_val_score(pipeline, X1, y, cv=CV, scoring="accuracy")

all_features = cross_val_score(pipeline, X, y, cv=CV, scoring="f1_macro")
all_features_accuracy = cross_val_score(pipeline, X, y, cv=CV, scoring="accuracy")

(
    only2features.mean(),
    only2features_accuracy.mean(),
    all_features.mean(),
    all_features_accuracy.mean(),
)


(np.float64(0.9490041688174017),
 np.float64(0.9506454166810092),
 np.float64(0.8000278908508525),
 np.float64(0.8236595412665082))

> **Вывод:** гипотеза подтвердилась. `Height` и `Weight` дали `macro-F1 = 0.949` и `accuracy = 0.951`, все признаки -- `0.800` и `0.824`.


### 2. Добавление `BMI` и `Weight / Height` повысит качество

**Гипотеза:** новые признаки, отражающие соотношение веса и роста, улучшат качество KNN.


In [9]:
X2 = X.copy()

X2["BMI"] = X2["Weight"] / (X2["Height"])**2
BMIfeatures = cross_val_score(pipeline, X2, y, cv=CV, scoring="f1_macro")
BMIfeatures_accuracy = cross_val_score(pipeline, X2, y, cv=CV, scoring="accuracy")

X2["WH"] = X2["Weight"] / X2["Height"]  # задумка: смотрим коэффициент наклона облака из EDA
BMIplusWHfeatures = cross_val_score(pipeline, X2, y, cv=CV, scoring="f1_macro")
BMIplusWHfeatures_accuracy = cross_val_score(
    pipeline,
    X2,
    y,
    cv=CV,
    scoring="accuracy",
)

(
    all_features.mean(),
    all_features_accuracy.mean(),
    BMIfeatures.mean(),
    BMIfeatures_accuracy.mean(),
    BMIplusWHfeatures.mean(),
    BMIplusWHfeatures_accuracy.mean(),
)


(np.float64(0.8000278908508525),
 np.float64(0.8236595412665082),
 np.float64(0.8350743824687245),
 np.float64(0.8538455360113822),
 np.float64(0.85707234988693),
 np.float64(0.8720583341938889))

> **Вывод:** гипотеза подтвердилась. `BMI` повысил `macro-F1` с `0.800` до `0.835`, а добавление обоих признаков -- до `0.857`.


### 3. `Weight`, `Height` и `BMI` окажутся наиболее важными признаками

**Гипотеза:** антропометрические признаки займут верхние позиции по permutation importance дерева решений.


In [10]:
X3 = X.copy()
X3["BMI"] = X3["Weight"] / (X3["Height"])**2

X_train3, X_test3, y_train3, y_test3 = train_test_split(
    X3,
    y,
    test_size=0.25,
    stratify=y,
    random_state=SEED,
)

decision_tree = make_pipeline(
    transformer,
    DecisionTreeClassifier(random_state=SEED),
)
decision_tree.fit(X_train3, y_train3)

permutation = permutation_importance(
    decision_tree,
    X_test3,
    y_test3,
    scoring="f1_macro",
    n_repeats=10,
    random_state=SEED,
)

importances = list(zip(X3.columns, permutation.importances_mean))
importances.sort(key=lambda x: x[1], reverse=True)

most_important = [i[0] for i in importances[:5]]

most_important, importances[:10]


(['BMI', 'Weight', 'Gender', 'NCP', 'TUE'],
 [('BMI', np.float64(0.7789330124344243)),
  ('Weight', np.float64(0.14801398747680655)),
  ('Gender', np.float64(0.12943678272707543)),
  ('NCP', np.float64(0.018566035323110897)),
  ('TUE', np.float64(0.017830053450139716)),
  ('MTRANS', np.float64(0.008863778292624991)),
  ('Height', np.float64(0.006081146243642454)),
  ('FAVC', np.float64(0.005482379196536524)),
  ('family_history_with_overweight', np.float64(0.003373224408968434)),
  ('FCVC', np.float64(0.002518803082710053))])

> **Вывод:** гипотеза подтвердилась частично. `BMI` и `Weight` заняли первые два места, но `Height` оказался шестым.


### 4. Пяти лучших признаков будет достаточно для сохранения качества

**Гипотеза:** KNN на пяти лучших признаках не уступит KNN на полном наборе с добавленным `BMI`.


In [11]:
X4 = X3[most_important]

mostimportantfeatures = cross_val_score(
    pipeline,
    X4,
    y,
    cv=CV,
    scoring="f1_macro",
)
mostimportantfeatures_accuracy = cross_val_score(
    pipeline,
    X4,
    y,
    cv=CV,
    scoring="accuracy",
)

(
    BMIfeatures.mean(),
    BMIfeatures_accuracy.mean(),
    mostimportantfeatures.mean(),
    mostimportantfeatures_accuracy.mean(),
)


(np.float64(0.8350743824687245),
 np.float64(0.8538455360113822),
 np.float64(0.9118896037867474),
 np.float64(0.9156689959037555))

> **Вывод:** гипотеза подтвердилась. Топ-5 повысил `macro-F1` с `0.835` до `0.913`, однако отбор признаков выполнен до кросс-валидации.


### 5. Удаление почти константных признаков не ухудшит качество

**Гипотеза:** удаление `SMOKE` и `SCC`, у которых одна категория преобладает более чем в 90% строк, не ухудшит или немного повысит качество.


In [12]:
X5 = X.drop(columns=["SCC", "SMOKE"])

noncontstantfeatures = cross_val_score(
    pipeline,
    X5,
    y,
    cv=CV,
    scoring="f1_macro",
)
noncontstantfeatures_accuracy = cross_val_score(
    pipeline,
    X5,
    y,
    cv=CV,
    scoring="accuracy",
)

(
    all_features.mean(),
    all_features_accuracy.mean(),
    noncontstantfeatures.mean(),
    noncontstantfeatures_accuracy.mean(),
)


(np.float64(0.8000278908508525),
 np.float64(0.8236595412665082),
 np.float64(0.8021540246410377),
 np.float64(0.8246210686952831))

> **Вывод:** эффект почти отсутствует. `macro-F1` вырос с `0.800` до `0.802`, `accuracy` -- с `0.824` до `0.825`.


### 6. Регуляризация не даст существенного прироста

**Гипотеза:** из-за слабой коррелированности числовых признаков L1-регуляризация и `RidgeClassifier` не дадут существенного прироста относительно обычной логистической регрессии.


In [13]:
X6 = X.copy()

pipeline_lr = make_pipeline(
    transformer,
    LogisticRegression(max_iter=5000, random_state=SEED),
)
pipeline_lasso = make_pipeline(
    transformer,
    LogisticRegression(l1_ratio=1, solver="saga", max_iter=5000, random_state=SEED),
)
pipeline_ridge = make_pipeline(
    transformer,
    RidgeClassifier(random_state=SEED),
)

condition_number = np.linalg.cond(StandardScaler().fit_transform(X6[numerical_features]))

lr = cross_val_score(pipeline_lr, X6, y, cv=CV, scoring="f1_macro")
lr_accuracy = cross_val_score(pipeline_lr, X6, y, cv=CV, scoring="accuracy")

lasso = cross_val_score(pipeline_lasso, X6, y, cv=CV, scoring="f1_macro")
lasso_accuracy = cross_val_score(pipeline_lasso, X6, y, cv=CV, scoring="accuracy")

ridge = cross_val_score(pipeline_ridge, X6, y, cv=CV, scoring="f1_macro")
ridge_accuracy = cross_val_score(pipeline_ridge, X6, y, cv=CV, scoring="accuracy")

(
    condition_number,
    lr.mean(),
    lr_accuracy.mean(),
    lasso.mean(),
    lasso_accuracy.mean(),
    ridge.mean(),
    ridge_accuracy.mean(),
)


(np.float64(2.2214657122491257),
 np.float64(0.8812514844781184),
 np.float64(0.8859545856138056),
 np.float64(0.9608037406551198),
 np.float64(0.9621481761958854),
 np.float64(0.5996751938530684),
 np.float64(0.6368099778550366))

> **Вывод:** гипотеза не подтвердилась. При числе обусловленности `2.22` L1 повысила `macro-F1` с `0.881` до `0.961`, а `RidgeClassifier` снизил его до `0.600`.


### 7. Сохранение потенциальных выбросов даст лучшее качество

**Гипотеза:** удаление IQR-выбросов из train приведёт к потере полезной информации и снижению качества на неизменённом test.


In [14]:
X7 = X.copy()

X_train7, X_test7, y_train7, y_test7 = train_test_split(
    X7,
    y,
    test_size=0.25,
    stratify=y,
    random_state=SEED,
)

outlier_features = ["Age", "Height", "Weight"]

q1 = X_train7[outlier_features].quantile(0.25)
q3 = X_train7[outlier_features].quantile(0.75)
iqr = q3 - q1

lower_bounds = q1 - 1.5 * iqr
upper_bounds = q3 + 1.5 * iqr

train_outlier_mask = (
    (X_train7[outlier_features] < lower_bounds)
    | (X_train7[outlier_features] > upper_bounds)
).any(axis=1)

X_train7_cleaned = X_train7.loc[~train_outlier_mask]
y_train7_cleaned = y_train7.loc[~train_outlier_mask]

baseline_model = clone(pipeline)
cleaned_model = clone(pipeline)

baseline_model.fit(X_train7, y_train7)
cleaned_model.fit(X_train7_cleaned, y_train7_cleaned)

baseline_predictions = baseline_model.predict(X_test7)
cleaned_predictions = cleaned_model.predict(X_test7)

baseline_f1 = f1_score(y_test7, baseline_predictions, average="macro")
baseline_accuracy = accuracy_score(y_test7, baseline_predictions)

cleaned_f1 = f1_score(y_test7, cleaned_predictions, average="macro")
cleaned_accuracy = accuracy_score(y_test7, cleaned_predictions)

(
    train_outlier_mask.sum(),
    baseline_f1,
    baseline_accuracy,
    cleaned_f1,
    cleaned_accuracy,
)


(np.int64(118),
 0.7875545051150444,
 0.814176245210728,
 0.7534664565140006,
 0.7739463601532567)

> **Вывод:** гипотеза подтвердилась. После удаления 118 объектов из train `accuracy` снизилась с `0.814` до `0.774`, `macro-F1` -- с `0.788` до `0.753`.
